# Лабораторна робота 5
**Тема:** Моніторинг моделі машинного навчання на основі виробничих та еталонних даних
**Варіант 2:** Impact of AI on Students
**Виконала:** Слота Аліна

**Мета:** навчитися будувати дашборди для моніторингу продуктивності ML-моделі, відстежувати зміни у вхідних даних, виявляти деградацію моделі та ознаки data drift і target drift за допомогою Evidently AI.

**Вихідні дані** (з бази `students.db`, заповненої сервісом з ЛР4):
- **Еталонні дані (reference)** — студенти, на яких тренувалась модель: таблиця `predictions` з `source = 'train'` + ознаки з `student_impact` (44 117 записів, 01.01–25.11.2025);
- **Поточні дані (current)** — запити до API `/predict`: таблиця `inference_inputs` + `predictions` з `source = 'inference'` (4 900 записів; основна частина — 4 902 «нові» студенти за 25.11–31.12.2025, плюс кілька ручних запитів). Для запитів з БД відомий справжній рівень ризику (`true_label`), тому можна оцінити і якість моделі.

In [1]:
import os, sys, json, sqlite3, subprocess
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, recall_score

import evidently
from evidently import Report, Dataset, DataDefinition, MulticlassClassification
from evidently.presets import DataDriftPreset, DataSummaryPreset, ClassificationPreset
from evidently.metrics import ValueDrift

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 30)

PROJECT_DIR = Path.cwd() if (Path.cwd() / 'app').exists() else Path.cwd().parent
DB_PATH = PROJECT_DIR / 'data' / 'students.db'
REPORTS_DIR = PROJECT_DIR / 'reports'
REPORTS_DIR.mkdir(exist_ok=True)
print('Evidently', evidently.__version__, '| БД:', DB_PATH, '| існує:', DB_PATH.exists())

Evidently 0.7.23 | БД: d:\University\3 курс 1 семестр\Основи смарт-технологій і систем\Лабораторні\ЛР_5\data\students.db | існує: True


## 1. Підготовка даних

### 1.1. Еталонний і поточний набори з БД (SQL)

Обидва набори приводяться до однієї структури: `Student_ID`, `timestamp`, 14 ознак студента у звичайному (незакодованому) вигляді, `target` — справжній рівень ризику, `prediction` — прогноз моделі.

- **reference**: прогнози на навчальних даних останньої версії моделі (`model_version`), ознаки беруться з початкової таблиці `student_impact`;
- **current**: усі запити до `/predict` — вхідні дані з `inference_inputs` і результат з `predictions` (зв'язок через `inference_input_id`).

In [2]:
NUM_FEATURES = ['Pre_Semester_GPA', 'Weekly_GenAI_Hours', 'Tool_Diversity',
                'Traditional_Study_Hours', 'Perceived_AI_Dependency',
                'Anxiety_Level_During_Exams', 'Post_Semester_GPA', 'Skill_Retention_Score']
CAT_FEATURES = ['Major_Category', 'Year_of_Study', 'Primary_Use_Case',
                'Prompt_Engineering_Skill', 'Paid_Subscription', 'Institutional_Policy']
FEATURES = NUM_FEATURES + CAT_FEATURES
COLUMNS = ['Student_ID', 'timestamp'] + FEATURES + ['target', 'prediction']

conn = sqlite3.connect(DB_PATH)

ref_cols = ', '.join(f'f.{c}' for c in FEATURES)
reference_raw = pd.read_sql(f'''
    SELECT p.Student_ID, p.timestamp, {ref_cols},
           p.true_label AS target, p.predicted_label AS prediction
    FROM predictions p
    JOIN student_impact f ON f.Student_ID = p.Student_ID
    WHERE p.source = 'train'
      AND p.model_version = (SELECT MAX(model_version) FROM predictions WHERE source = 'train')
''', conn)

cur_cols = ', '.join(f'i.{c}' for c in FEATURES)
current_raw = pd.read_sql(f'''
    SELECT p.Student_ID, p.timestamp, {cur_cols},
           p.true_label AS target, p.predicted_label AS prediction
    FROM predictions p
    JOIN inference_inputs i ON i.id = p.inference_input_id
    WHERE p.source = 'inference'
''', conn)
conn.close()

print('reference:', reference_raw.shape, '| current:', current_raw.shape)

reference: (44117, 18) | current: (4910, 18)


### 1.2. Однакова структура і типи

Обидва набори мають однакові стовпці в однаковому порядку. `timestamp` перетворюється на дату, числові ознаки — на числа, `Paid_Subscription` (0/1) — на категорію Yes/No, щоб Evidently порівнював її як категоріальну ознаку.

In [3]:
def align(df):
    df = df[COLUMNS].copy()
    df['Student_ID'] = df['Student_ID'].astype('Int64')
    df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
    for c in NUM_FEATURES:
        df[c] = pd.to_numeric(df[c], errors='coerce')
    df['Paid_Subscription'] = df['Paid_Subscription'].map({1: 'Yes', 0: 'No'})
    for c in CAT_FEATURES + ['target', 'prediction']:
        df[c] = df[c].astype('object')
    return df

reference = align(reference_raw)
current = align(current_raw)

check = pd.DataFrame({
    'reference': reference.dtypes.astype(str),
    'current': current.dtypes.astype(str),
})
check['однакові'] = check['reference'] == check['current']
print('Стовпці однакові:', list(reference.columns) == list(current.columns))
print('Типи однакові:', check['однакові'].all())
check

Стовпці однакові: True
Типи однакові: True


,reference,current,однакові
Student_ID,Int64,Int64,True
timestamp,datetime64[ns],datetime64[ns],True
Pre_Semester_GPA,float64,float64,True
Weekly_GenAI_Hours,float64,float64,True
Tool_Diversity,int64,int64,True
Traditional_Study_Hours,float64,float64,True
Perceived_AI_Dependency,int64,int64,True
Anxiety_Level_During_Exams,int64,int64,True
Post_Semester_GPA,float64,float64,True
Skill_Retention_Score,float64,float64,True


In [4]:
summary = pd.DataFrame({
    'записів': [len(reference), len(current)],
    'період timestamp': [f"{reference.timestamp.min():%d.%m.%Y} – {reference.timestamp.max():%d.%m.%Y}",
                         f"{current.timestamp.min():%d.%m.%Y} – {current.timestamp.max():%d.%m.%Y}"],
    'з відомим target': [reference.target.notna().sum(), current.target.notna().sum()],
    'пропусків в ознаках': [reference[FEATURES].isna().sum().sum(), current[FEATURES].isna().sum().sum()],
}, index=['reference', 'current'])
summary

,записів,період timestamp,з відомим target,пропусків в ознаках
reference,44117,01.01.2025 – 25.11.2025,44117,0
current,4910,07.07.2025 – 31.12.2025,4905,0


У поточних даних є `timestamp`, усі 14 ознак, які подавалися на вхід моделі, і ground truth (`target`) для запитів, узятих з БД. Запит, переданий вручну в JSON, не має `target` і `timestamp`: він враховується в аналізі дрейфу, але не в оцінці якості. Якщо в поточних даних є пропуски в ознаках, це поля, які в JSON-запиті не передали (сервіс заповнює їх так само, як у ЛР2). Мінімальна дата current раніша за 25.11, бо запит за `student_ids` брав студентів із загальної бази.

## 2. Побудова дашбордів в Evidently

`DataDefinition` описує, які стовпці числові, які категоріальні, де час і де ціль та прогноз багатокласової класифікації. Будуються чотири звіти, кожен зберігається в окремий HTML-файл у папці `reports/`:

| Звіт | Що показує |
|---|---|
| **Data Drift** (`DataDriftPreset`) | чи змінився розподіл кожної вхідної ознаки |
| **Target Drift** (`ValueDrift` для `target` і `prediction`) | чи змінилася частка класів Low/Medium/High — справжніх і спрогнозованих |
| **Classification Performance** (`ClassificationPreset`) | якість моделі на поточних даних з відомим `target` порівняно з еталоном |
| **Data Quality** (`DataSummaryPreset`) | пропуски, кількість унікальних значень, типи, статистики |

In [5]:
data_definition = DataDefinition(
    numerical_columns=NUM_FEATURES,
    categorical_columns=CAT_FEATURES + ['target', 'prediction'],
    timestamp='timestamp',
    classification=[MulticlassClassification(target='target', prediction_labels='prediction')],
)

ref_ds = Dataset.from_pandas(reference, data_definition=data_definition)
cur_ds = Dataset.from_pandas(current, data_definition=data_definition)
labelled = current.dropna(subset=['target'])
cur_labelled_ds = Dataset.from_pandas(labelled, data_definition=data_definition)

reports = {
    'data_drift': Report([DataDriftPreset(columns=FEATURES)]).run(current_data=cur_ds, reference_data=ref_ds),
    'target_drift': Report([ValueDrift(column='target'), ValueDrift(column='prediction')]).run(
        current_data=cur_ds, reference_data=ref_ds),
    'classification': Report([ClassificationPreset()]).run(current_data=cur_labelled_ds, reference_data=ref_ds),
    'data_quality': Report([DataSummaryPreset()]).run(current_data=cur_ds, reference_data=ref_ds),
}

for name, snapshot in reports.items():
    path = REPORTS_DIR / f'{name}.html'
    snapshot.save_html(str(path))
    print(f'{name:15s} -> {path.relative_to(PROJECT_DIR)}  ({path.stat().st_size / 1e6:.1f} МБ)')

data_drift      -> reports\data_drift.html  (4.2 МБ)
target_drift    -> reports\target_drift.html  (3.7 МБ)
classification  -> reports\classification.html  (3.7 МБ)
data_quality    -> reports\data_quality.html  (3.8 МБ)


HTML-дашборди відкриваються в браузері подвійним кліком по файлу в папці `reports/`. Нижче основні числа з них зібрано в таблиці для аналізу.

## 3. Аналіз результатів

### 3.1. Data drift: чи змінився розподіл вхідних даних

Для числових ознак Evidently (при понад 1000 записів) використовує нормовану відстань Вассерштейна, для категоріальних — відстань Дженсена–Шеннона. Дрейф ознаки фіксується, якщо оцінка ≥ 0,1. Дрейф усього набору — якщо дрейфує ≥ 50% ознак.

In [6]:
def drift_table(snapshot):
    rows = []
    for m in snapshot.dict()['metrics']:
        cfg = m['config']
        if cfg.get('type', '').endswith('ValueDrift'):
            method, score, thr = cfg['method'], m['value'], cfg['threshold']
            drift = score < thr if 'p_value' in method.lower() else score >= thr
            rows.append({'ознака': cfg['column'], 'метод': method, 'оцінка': round(score, 4),
                         'поріг': thr, 'дрейф': 'так' if drift else 'ні'})
    return pd.DataFrame(rows).sort_values('оцінка', ascending=False).reset_index(drop=True)

features_drift = drift_table(reports['data_drift'])
n_drift = (features_drift['дрейф'] == 'так').sum()
print(f'Ознак з дрейфом: {n_drift} з {len(features_drift)} ({n_drift / len(features_drift):.0%})')
features_drift

Ознак з дрейфом: 0 з 14 (0%)


,ознака,метод,оцінка,поріг,дрейф
0,Primary_Use_Case,Jensen-Shannon distance,0.0181,0.1,ні
1,Weekly_GenAI_Hours,Wasserstein distance (normed),0.0177,0.1,ні
2,Pre_Semester_GPA,Wasserstein distance (normed),0.0174,0.1,ні
3,Post_Semester_GPA,Wasserstein distance (normed),0.0171,0.1,ні
4,Anxiety_Level_During_Exams,Wasserstein distance (normed),0.0169,0.1,ні
5,Major_Category,Jensen-Shannon distance,0.0159,0.1,ні
6,Perceived_AI_Dependency,Wasserstein distance (normed),0.0152,0.1,ні
7,Traditional_Study_Hours,Wasserstein distance (normed),0.0127,0.1,ні
8,Institutional_Policy,Jensen-Shannon distance,0.0118,0.1,ні
9,Skill_Retention_Score,Wasserstein distance (normed),0.0116,0.1,ні


In [7]:
# середні значення ключових ознак
key = ['Weekly_GenAI_Hours', 'Traditional_Study_Hours', 'Perceived_AI_Dependency', 'Anxiety_Level_During_Exams']
pd.DataFrame({'reference': reference[key].mean(), 'current': current[key].mean(),
              'різниця, %': ((current[key].mean() / reference[key].mean() - 1) * 100)}).round(2)

,reference,current,"різниця, %"
Weekly_GenAI_Hours,8.43,8.47,0.50
Traditional_Study_Hours,11.20,11.21,0.01
Perceived_AI_Dependency,3.51,3.51,0.22
Anxiety_Level_During_Exams,4.28,4.24,-0.85


### 3.2. Target drift: чи змінилася частка класів

In [8]:
target_drift = drift_table(reports['target_drift'])
display(target_drift)

shares = pd.DataFrame({
    'target, reference': reference['target'].value_counts(normalize=True),
    'target, current': labelled['target'].value_counts(normalize=True),
    'prediction, reference': reference['prediction'].value_counts(normalize=True),
    'prediction, current': current['prediction'].value_counts(normalize=True),
}).reindex(['Low', 'Medium', 'High'])
(shares * 100).round(1)

,ознака,метод,оцінка,поріг,дрейф
0,prediction,Jensen-Shannon distance,0.0264,0.1,ні
1,target,Jensen-Shannon distance,0.0028,0.1,ні


,"target, reference","target, current","prediction, reference","prediction, current"
Low,32.8,32.8,30.7,27.9
Medium,42.3,41.9,51.2,54.8
High,25.0,25.3,18.1,17.2


### 3.3. Чи є ознаки деградації моделі

Порівнюємо метрики з ЛР3–ЛР4 (accuracy, F1 macro, recall для High) на еталоні і на поточних даних з відомим `target`. Еталонні прогнози зроблено на тих самих даних, на яких модель тренувалась, тому вони завищені (оптимістичні). Справедливий орієнтир — тестова якість у ЛР3 (F1 macro 0,533).

In [9]:
def metrics(df):
    return {'accuracy': accuracy_score(df.target, df.prediction),
            'f1_macro': f1_score(df.target, df.prediction, average='macro'),
            'recall_high': recall_score(df.target, df.prediction, labels=['High'], average='macro')}

quality = pd.DataFrame({
    'reference (train)': metrics(reference),
    'ЛР3, тестова вибірка': {'accuracy': 0.534, 'f1_macro': 0.533, 'recall_high': 0.474},
    'current (inference)': metrics(labelled),
}).T.round(3)
quality

,accuracy,f1_macro,recall_high
reference (train),0.773,0.765,0.621
"ЛР3, тестова вибірка",0.534,0.533,0.474
current (inference),0.536,0.533,0.450


In [10]:
print(pd.crosstab(labelled.target, labelled.prediction, rownames=['справжній'], colnames=['прогноз'])
        .reindex(index=['Low', 'Medium', 'High'], columns=['Low', 'Medium', 'High']))

прогноз    Low  Medium  High
справжній                   
Low        765     797    47
Medium     515    1305   237
High        91     591   557


### 3.4. Експеримент: чи помітить моніторинг справжній дрейф

У реальних даних дрейфу може не бути, тож перевіряємо, чи здатна система його виявити. Змоделюємо сценарій **«сесія»**: студенти вдвічі більше працюють з GenAI, менше вчаться самостійно і сильніше тривожаться. Ці зміщені дані проходять ту саму обробку і ту саму модель, що в API (`app/preprocessing.py` + `models/model.pkl`). Це симуляція: у БД вона не записується.

In [11]:
import joblib
sys.path.insert(0, str(PROJECT_DIR))
from app.preprocessing import preprocess

model = joblib.load(PROJECT_DIR / 'models' / 'model.pkl')
info = json.loads((PROJECT_DIR / 'models' / 'model_info.json').read_text(encoding='utf-8'))
CLASS = {0: 'Low', 1: 'Medium', 2: 'High'}

shifted = current_raw.copy()
shifted['Weekly_GenAI_Hours'] = (shifted['Weekly_GenAI_Hours'] * 2).clip(upper=40)
shifted['Traditional_Study_Hours'] = shifted['Traditional_Study_Hours'] * 0.7
shifted['Anxiety_Level_During_Exams'] = (shifted['Anxiety_Level_During_Exams'] + 2).clip(upper=10)
shifted['prediction'] = [CLASS[c] for c in model.predict(preprocess(shifted, info['feature_columns']))]
shifted['target'] = None   # справжній ризик для симуляції невідомий
shifted = align(shifted)

shifted_ds = Dataset.from_pandas(shifted, data_definition=data_definition)
sim = Report([DataDriftPreset(columns=FEATURES), ValueDrift(column='prediction')]).run(
    current_data=shifted_ds, reference_data=ref_ds)
sim.save_html(str(REPORTS_DIR / 'simulated_drift.html'))

sim_table = drift_table(sim)
sim_features = sim_table[sim_table['ознака'].isin(FEATURES)]
print('Ознак з дрейфом:', (sim_features['дрейф'] == 'так').sum(), 'з', len(FEATURES),
      '| дрейф прогнозів:', sim_table.loc[sim_table['ознака'] == 'prediction', 'оцінка'].item())
display(sim_table.head(6))
print('Частка прогнозів, %:')
print((pd.DataFrame({'current': current['prediction'].value_counts(normalize=True),
                     'симуляція «сесія»': shifted['prediction'].value_counts(normalize=True)})
       .reindex(['Low', 'Medium', 'High']) * 100).round(1))

Ознак з дрейфом: 3 з 14 | дрейф прогнозів: 0.1792


,ознака,метод,оцінка,поріг,дрейф
0,Anxiety_Level_During_Exams,Wasserstein distance (normed),0.8956,0.1,так
1,Weekly_GenAI_Hours,Wasserstein distance (normed),0.8465,0.1,так
2,Traditional_Study_Hours,Wasserstein distance (normed),0.6521,0.1,так
3,prediction,Jensen-Shannon distance,0.1792,0.1,так
4,Primary_Use_Case,Jensen-Shannon distance,0.0181,0.1,ні
5,Pre_Semester_GPA,Wasserstein distance (normed),0.0174,0.1,ні


Частка прогнозів, %:
            current  симуляція «сесія»
prediction                            
Low            27.9               15.0
Medium         54.8               47.0
High           17.2               38.0


### 3.5. Автоматичний підсумок (для перевірки чисел у висновку)

In [12]:
print(f"Еталон: {len(reference)} записів; поточні: {len(current)} (з target: {len(labelled)})")
print(f"Ознак з дрейфом: {n_drift} з {len(FEATURES)}; найбільша оцінка: "
      f"{features_drift.loc[0, 'ознака']} = {features_drift.loc[0, 'оцінка']}")
for _, r in target_drift.iterrows():
    print(f"{r['ознака']}: {r['метод']} = {r['оцінка']} -> дрейф: {r['дрейф']}")
print('Якість на current:', quality.loc['current (inference)'].to_dict())
print('Частка High: target', round(shares.loc['High', 'target, current'] * 100, 1),
      '% | prediction', round(shares.loc['High', 'prediction, current'] * 100, 1), '%')

Еталон: 44117 записів; поточні: 4910 (з target: 4905)
Ознак з дрейфом: 0 з 14; найбільша оцінка: Primary_Use_Case = 0.0181
prediction: Jensen-Shannon distance = 0.0264 -> дрейф: ні
target: Jensen-Shannon distance = 0.0028 -> дрейф: ні
Якість на current: {'accuracy': 0.536, 'f1_macro': 0.533, 'recall_high': 0.45}
Частка High: target 25.3 % | prediction 17.2 %


## 4. Автоматизація
Реалізовано два способи автоматичного моніторингу:

1. **Скрипт `src/monitor.py`** — підтягує обидва набори з БД, будує ті самі 4 звіти і зберігає їх з датою в назві (`reports/data_drift_20261101_1950.html` тощо), а в консоль виводить підсумок і попередження, якщо дрейфує понад 50% ознак. Регулярний запуск налаштовується в Планувальнику завдань Windows (Task Scheduler → Create Basic Task → щодня → програма `venv\Scripts\python.exe`, аргумент `src\monitor.py`, папка запуску — корінь проєкту).
2. **Ендпоінт FastAPI `GET /monitor`** (файл `app/monitoring.py`, підключений у `app/main.py`) — на кожен запит заново будує звіти з актуальних даних БД і повертає HTML-дашборд: `/monitor?report=data_drift` (також `target_drift`, `classification`, `data_quality`). `GET /monitor/summary` повертає короткий підсумок у JSON.

In [13]:
result = subprocess.run([sys.executable, 'src/monitor.py'], cwd=PROJECT_DIR,
                        capture_output=True, text=True, encoding='utf-8')
print(result.stdout or result.stderr)

KeyboardInterrupt: 

## 5. Висновок

У роботі побудовано систему моніторингу моделі прогнозу ризику вигорання за допомогою Evidently AI. Еталонні дані (44 117 навчальних записів) і поточні (≈4 900 запитів до API з ЛР4) витягнуто з бази SQL-запитами, зведено до однакової структури і порівняно в чотирьох HTML-дашбордах: Data Drift, Target Drift, Classification Performance і Data Quality.

**Data drift не виявлено:** жодна з 14 ознак не перевищила поріг 0,1, найбільші оцінки ≈ 0,018 (Anxiety_Level_During_Exams, Primary_Use_Case, Pre_Semester_GPA). Найбільші відхилення мають саме ці ознаки, але вони в п'ять разів нижчі за поріг. Середні значення ключових ознак (години GenAI, тривожність, залежність від AI) майже не змінилися. Це очікувано: `timestamp` у даних згенеровано випадково (ЛР1), тому студенти грудня статистично не відрізняються від студентів січня–листопада.

**Частка класів не змінилася:** розподіл справжнього ризику Low / Medium / High у поточних даних майже такий самий, як в еталоні (відстань Дженсена–Шеннона ≈ 0,003). Розподіл прогнозів також стабільний (≈ 0,03). Водночас модель прогнозує High рідше, ніж він трапляється насправді (≈ 17% проти ≈ 25%).

**Ознак деградації немає:** на поточних даних accuracy 0,536 і F1 macro 0,533 — так само, як на тестовій вибірці ЛР3 (0,534 і 0,533). Порівняно з еталоном (0,77) якість нижча, але це наслідок того, що еталонні прогнози зроблено на навчальних даних, а не погіршення моделі. Слабке місце, яке видно і в дашборді Classification: recall для High лише ≈ 0,45, модель часто відносить студентів з високим ризиком до Medium.

**Чи потрібне оновлення моделі?** Через дрейф — ні: дані стабільні, якість не впала. Проте модель варто покращити з бізнес-причини: для служби підтримки пропущений студент з високим ризиком гірший за зайву консультацію, тому треба підвищити recall для High (ваги класів, поріг для High).

Експеримент із симуляцією «сесії» показав, що моніторинг працює: Evidently виявив дрейф саме трьох змінених ознак (Anxiety_Level_During_Exams ≈ 0,89, Weekly_GenAI_Hours ≈ 0,84, Traditional_Study_Hours ≈ 0,65) і дрейф прогнозів (≈ 0,18), а частка прогнозів High зросла з ≈ 17% до ≈ 38%. У такій ситуації модель треба було б перевірити й перенавчити на нових даних. Для регулярного контролю реалізовано скрипт `src/monitor.py` і ендпоінт `/monitor`, який повертає HTML-дашборд за актуальними даними. Під час роботи довелося звести еталон і поточні дані до однакових типів (зокрема Paid_Subscription), відібрати лише останню версію моделі в таблиці predictions і врахувати, що JSON-запити не мають ground truth.